# 02 · pandas
NumPy nous a aidés à calculer avec des arrays. pandas ajoute des **étiquettes de lignes et de colonnes**, ainsi que des outils pour travailler avec des tables dont les colonnes peuvent avoir des types différents. Notre démarche est **inspecter → sélectionner → résumer → visualiser**.

Nous commencerons avec trois fleurs à titre d’illustration, puis explorerons le dataset Iris : 150 fleurs, quatre mesures en centimètres et trois espèces. Iris est fourni avec scikit-learn ; `load_iris` ne télécharge aucune donnée. Le prochain notebook utilise deux de ces espèces pour la classification.

**Comment utiliser ce notebook :** prédisez un résultat, exécutez la cellule, puis expliquez ce que représente une ligne du résultat. `display(...)` affiche une table mise en forme dans Jupyter ou Colab. Le code des graphiques est fourni ; concentrez-vous sur la table derrière chaque graphique. Exécutez le notebook du début à la fin avec NumPy, pandas, Matplotlib et scikit-learn installés. Aucun fichier de données supplémentaire n’est nécessaire.

**En classe :** suivez les sections A–E, y compris les exercices E et F, les tables de résumé, le diagramme en barres et le nuage de points.
**Après le cours :** la section **Exos pour vous** permet de pratiquer à votre rythme.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from IPython.display import display

# Display precision only; stored values keep their full precision.
pd.set_option("display.precision", 3)

## A. Construire une table facile à lire
Un **DataFrame** est une table 2D. Ses colonnes ont des noms et ses lignes ont un **index**. Un dictionnaire de listes de même longueur est une façon de le créer : les clés du dictionnaire deviennent les noms des colonnes.

Ces trois fleurs ont des valeurs illustratives. Les étiquettes de lignes sont des identifiants, pas des mesures. Remarquez que `species` contient du texte, tandis que les autres colonnes contiennent des nombres.

In [ ]:
mini = pd.DataFrame({
    "species": ["setosa", "versicolor", "virginica"],
    "petal_length_cm": [1.4, 4.7, 6.0],
    "petal_width_cm": [0.2, 1.4, 2.5],
}, index=["flower_a", "flower_b", "flower_c"])
display(mini)

### Une colonne : Series ou DataFrame ?
Une **Series** est une colonne 1D munie d’étiquettes. Une liste de noms de colonnes conserve un DataFrame 2D, même si cette liste ne contient qu’un seul nom.

| Expression | Résultat | Forme |
|---|---|---|
| `mini["petal_length_cm"]` | Series | `(3,)` |
| `mini[["petal_length_cm"]]` | DataFrame | `(3, 1)` |

Cela ressemble au maintien ou à la suppression d’une dimension dans NumPy, mais les étiquettes de lignes restent associées aux valeurs.

In [ ]:
lengths = mini["petal_length_cm"]
length_table = mini[["petal_length_cm"]]
print("Series shape:", lengths.shape)
display(lengths)
print("DataFrame shape:", length_table.shape)
display(length_table)

## B. Inspecter avant de calculer
Chargez la table Iris complète et convertissez les codes numériques de `target` en noms d’espèces. Le code numérique est une **catégorie**, pas une mesure dont on doit calculer la moyenne. Nous sélectionnerons explicitement les colonnes de mesures pour les statistiques.

`head()` affiche les cinq premières lignes. Ici, ces lignes correspondent toutes à setosa : elles ne montrent donc pas toute la diversité du dataset.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()

# Iris utilise trois codes numériques. Ce dictionnaire donne le nom de chaque code.
species_names = {0: "setosa", 1: "versicolor", 2: "virginica"}
# map cherche chaque code cible dans le dictionnaire pour créer une colonne de noms.
df["species"] = df["target"].map(species_names)
display(df.head())

| Outil | Question à laquelle il répond |
|---|---|
| `df.shape` | Combien de lignes et de colonnes ? |
| `df.info()` | Quels sont les types des colonnes et les nombres de valeurs non manquantes ? |
| `df.describe()` | Quelles sont les plages de valeurs et les statistiques descriptives numériques ? |
| `df.isna().sum()` | Combien de valeurs manquantes y a-t-il dans chaque colonne ? |

**Lisez la sortie :** attendez-vous à 150 lignes et six colonnes : quatre mesures, un code cible et un nom d’espèce. Dans `describe`, `50%` est la médiane ; `25%` et `75%` délimitent la moitié centrale des valeurs. `.round(2)` ci-dessous arrondit le résumé affiché, pas les mesures dans `df`.

In [ ]:
print("Shape:", df.shape)

df.info()

In [ ]:
measurement_columns = iris.feature_names
print("Measurement columns:", measurement_columns)
measurements = df[measurement_columns]
# Calculez les statistiques, puis arrondissez le résumé pour l’affichage.
measurement_summary = measurements.describe()
rounded_summary = measurement_summary.round(2)
display(rounded_summary)

## C. Sélectionner des lignes et des colonnes
Les deux sélecteurs utilisent **`[rows, columns]`**, comme dans les exemples NumPy.

| Sélecteur | Utilise | Borne de fin du slice |
|---|---|---|
| `.loc` | Des étiquettes de lignes/colonnes, ou un masque booléen | Inclut l’étiquette de fin |
| `.iloc` | Des positions entières | Exclut la position de fin |

Comparez les deux résultats ci-dessous. Sur cet index ordonné, les deux sélectionnent les deux premières fleurs et les mêmes deux colonnes. `flower_b` est une étiquette ; la position `1` est l’endroit où cette ligne se trouve actuellement. Un tri peut changer les positions sans changer les étiquettes.

In [ ]:
# reminder mini
print("Mini DataFrame:")
display(mini)

print("Selection:")
display(mini.loc["flower_a":"flower_b", ["species", "petal_width_cm"]])
display(mini.iloc[:2, [0, 2]])

### Transformer une question en masque
Quelles fleurs d’Iris ont une longueur de pétale supérieure à `5` cm ? La comparaison produit une valeur booléenne par ligne. `.loc` utilise ce masque pour conserver les lignes retenues et les colonnes indiquées. `sort_values` place ensuite les pétales les plus longs en premier.

Pour combiner des conditions, utilisez `&` pour **et** ou `|` pour **ou**, en plaçant chaque comparaison entre parenthèses. Utilisez `.isna()` pour détecter les valeurs manquantes.

In [ ]:
long_mask = df["petal length (cm)"] > 5
selected = df.loc[long_mask, ["species", "petal length (cm)"]]
ranked = selected.sort_values("petal length (cm)", ascending=False)
print("Rows selected:", len(selected))
display(ranked.head())

## D. Des lignes individuelles à une table récapitulative
`value_counts()` compte les lignes de chaque catégorie. `groupby("species")` répartit les fleurs par espèce ; sélectionner deux colonnes de mesures et appeler `.mean()` calcule une moyenne par espèce et par feature.

**Prédiction :** combien de lignes et de colonnes de mesures `petal_means` aura-t-il ? Une ligne de `df` est une fleur ; une ligne de `petal_means` est un résumé pour une espèce.

In [ ]:
# Comptez les fleurs de chaque espèce ; les noms des espèces deviennent l’index.
species_counts = df["species"].value_counts()

# Triez cet index par ordre alphabétique, puis transformez la Series en table à une colonne.
species_counts = species_counts.sort_index()
species_counts = species_counts.to_frame(name="flowers")

# Regroupez les lignes, sélectionnez les mesures, puis calculez les moyennes de chaque groupe.
petal_columns = ["petal length (cm)", "petal width (cm)"]
species_groups = df.groupby("species")
grouped_petals = species_groups[petal_columns]
petal_means = grouped_petals.mean()

display(species_counts)
display(petal_means.round(2))

### Le même résumé, sous forme de barres
pandas peut tracer un graphique directement à partir d’une table : l’index fournit les étiquettes des catégories et les colonnes numériques fournissent les hauteurs des barres.

**Interprétez le graphique :** chaque paire de barres provient d’une ligne de `petal_means`. Quelle espèce a les pétales les plus longs en moyenne ? Ces moyennes permettent-elles de savoir à quel point les mesures des fleurs individuelles se chevauchent ?

In [ ]:
ax = petal_means.plot.bar(figsize=(6.5, 3.5), rot=0,
                         color=["#0072B2", "#D55E00"])
ax.set(xlabel="Species", ylabel="Mean measurement (cm)",
       title="One summary row becomes a pair of bars")
ax.legend(["Petal length", "Petal width"])
plt.tight_layout()
plt.show()

### Exercice E · Sélection et agrégation (6 min)
1. Sélectionnez les fleurs virginica dont la largeur du pétale dépasse `2.0` cm.
2. Calculez leur longueur moyenne de pétale.
3. Calculez la longueur et la largeur moyennes des pétales pour **chaque espèce**, avec `groupby`.

**Vérifications :** 23 lignes sélectionnées ; longueur moyenne d’environ `5.76087` cm ; la table finale contient trois lignes et deux colonnes de mesures. Affichez les premières lignes de la table filtrée pour vérifier les deux conditions.

<details>
<summary>Indice</summary>

Commencez avec `(df["species"] == "virginica") & (df["petal width (cm)"] > 2.0)`. Sélectionnez avec `.loc`, puis calculez la moyenne des longueurs de pétale sélectionnées. L’étape 3 utilise tout `df`, pas seulement les fleurs filtrées.

</details>

In [ ]:
# 1. Build the two-condition mask and display the selected rows.

# 2. Compute their mean petal length.

# 3. Summarize both petal measurements for each species in the full df.

## E. Valeurs manquantes et observations individuelles
Iris ne contient aucune mesure manquante. Pour rendre ce cas visible, insérez une valeur manquante dans une **copie**. `.loc[row_label, column_label] = value` met à jour cette cellule. `np.nan` signifie qu’une valeur numérique est manquante ; cela ne signifie pas zéro.

`isna()` produit une table booléenne et `.sum()` compte les valeurs `True` par colonne. `dropna(subset=[...])` supprime les lignes où la mesure indiquée est manquante. **Prédiction :** combien de lignes restera-t-il ? Le `df` d’origine changera-t-il ?

In [ ]:
messy = df.copy()
messy.loc[0, "sepal width (cm)"] = np.nan

# isna marque les cellules manquantes par True ; sum les compte dans chaque colonne.
original_missing = df.isna()
edited_missing = messy.isna()
original_missing_counts = original_missing.sum()
edited_missing_counts = edited_missing.sum()

missing_counts = pd.DataFrame({
    "original": original_missing_counts,
    "after edit": edited_missing_counts,
})
display(missing_counts)
missing_width_mask = edited_missing["sepal width (cm)"]
missing_rows = messy.loc[missing_width_mask]
display(missing_rows)



In [ ]:
clean = messy.dropna(subset=["sepal width (cm)"])
print("Rows before / after:", len(messy), len(clean))
assert len(clean) == len(df) - 1
# Vérifiez que toutes les largeurs de sépale sont encore présentes dans la table d’origine.
original_widths_present = df["sepal width (cm)"].notna()
assert original_widths_present.all()
display(clean.isna().sum())

La suppression est un choix pour cet exercice. En pratique, cherchez pourquoi une valeur est manquante avant de décider de la supprimer ou de la remplacer. Les moyennes numériques ignorent les valeurs manquantes par défaut : `.count()` compte les valeurs observées, tandis que `len(...)` compte les lignes. Remplacer par zéro changerait la mesure et sa moyenne.

Supprimer des lignes conserve les étiquettes d’index restantes. Pour un modèle, toute imputation apprise à partir des données doit faire partie du processus d’entraînement.

### Exercice F · Un graphique et une phrase (5 min)
Exécutez le code du nuage de points. Chaque point représente une fleur : il complète donc les moyennes du diagramme en barres.

1. Décrivez une tendance et une limite.
2. Remplacez `x_feature` par `"sepal length (cm)"`, relancez et comparez. L’étiquette de l’axe horizontal suit automatiquement cette variable.

**Vérification :** la légende identifie les espèces et les deux axes indiquent les unités. Une association visible ne démontre ni un lien de causalité ni la performance sur des données futures.

In [ ]:
x_feature = "petal length (cm)"
y_feature = "petal width (cm)"
# Provided styles distinguish species with both colour and marker shape.
styles = {"setosa": ("o", "#0072B2"),
          "versicolor": ("s", "#D55E00"),
          "virginica": ("^", "#009E73")}
fig, ax = plt.subplots(figsize=(6.5, 3.8))
# Chaque tour de boucle donne un nom d’espèce et la table des fleurs de cette espèce.
for species, group in df.groupby("species"):
    # Récupérez les deux valeurs du style : la forme du marqueur, puis la couleur.
    marker, color = styles[species]
    ax.scatter(group[x_feature], group[y_feature], label=species,
               marker=marker, color=color, alpha=0.75)
ax.set(xlabel=x_feature, ylabel=y_feature,
       title="Individual flowers show variation within each species")
ax.legend()
fig.tight_layout()
plt.show()

**Vos observations :**

- Tendance : …
- Limite : …
- Après avoir changé la feature de l’axe horizontal : …

### Faites une pause et expliquez
En quoi une Series diffère-t-elle d’un DataFrame ? Qu’utilisent `.loc` et `.iloc` pour trouver les lignes ? Que représente une ligne d’un résumé obtenu par regroupement ? Que peut montrer le nuage de points qu’une moyenne ne peut pas montrer ?

## Exos pour vous

Ces exercices et explorations sont à faire après le cours, à votre rythme. Utilisez les tables préparées plus haut, modifiez les exemples fournis et expliquez ce que vous observez. Des indices et des solutions de référence se trouvent à la fin du notebook. Faites l’exercice 3 avant l’exercice 4 : l’exemple de doublon réutilise `sales`.

### 1. Une table de fréquences et un histogramme
Nous avons utilisé `value_counts` pour les catégories d’espèces. Pour une mesure numérique ayant beaucoup de valeurs distinctes, un histogramme regroupe les valeurs dans des **intervalles** appelés bins. Il compte les observations, pas les moyennes par espèce.

**À essayer :** changez `bins` de `12` à `6`. Le nombre d’observations change-t-il ? Le graphique regroupe les trois espèces ; quelle information cela masque-t-il ?

In [ ]:
display(species_counts)
petal_lengths = df["petal length (cm)"]
print("Petal lengths included:", petal_lengths.count())
ax = petal_lengths.plot.hist(
    bins=12, figsize=(6.5, 3.2), color="#0072B2", edgecolor="white")
ax.set(xlabel="Petal length (cm)", ylabel="Number of flowers",
       title="150 measurements grouped into intervals")
plt.tight_layout()
plt.show()

### 2. Mettre à jour une table et résoudre trois petits problèmes
Les opérations arithmétiques sur les colonnes s’appliquent à toutes les lignes, comme dans NumPy. Travaillez sur une copie, ajoutez une colonne dérivée, puis donnez-lui un nom plus clair. `.rename` et `.drop` renvoient de nouvelles tables par défaut : affectez donc leur résultat à une variable pour le conserver.

La nouvelle colonne en millimètres contient la même information dans une autre unité.

In [ ]:
enriched = mini.copy()
enriched["length_mm"] = enriched["petal_length_cm"] * 10
enriched = enriched.rename(columns={"length_mm": "petal_length_mm"})
display(enriched)
without_extra = enriched.drop(columns=["petal_length_mm"])
print("After dropping the extra column:", without_extra.shape)

Ces courtes tâches reprennent la section **DataFrame basics** de pandas puzzles (sélection, valeurs manquantes et mise à jour d’une ligne avec étiquette). Utilisez la copie de pratique ci-dessous ; la valeur manquante a été insérée volontairement.

1. Affichez uniquement la fleur dont la largeur est manquante.
2. Affichez les trois lignes triées par longueur de pétale, de la plus grande à la plus petite.
3. Notre donnée source confirme que la largeur manquante est `1.4` cm. Rétablissez cette seule valeur avec `.loc` et vérifiez qu’aucune largeur ne manque.

**Vérifications :** ligne `flower_b` ; ordre `flower_c`, `flower_b`, `flower_a` ; aucune largeur manquante après la correction. Le tri ne renomme pas les étiquettes de lignes.

In [ ]:
practice = mini.copy()
practice.loc["flower_b", "petal_width_cm"] = np.nan
display(practice)
# Find the missing row, sort by length, then restore the known width.

### 3. Lire une petite table de ventes et la pivoter
Explorez la lecture de CSV, le regroupement et le pivot avec six **commandes inventées**, assez peu nombreuses pour être vérifiées à la main. Chaque ligne est une commande, et `quantity` compte les articles.

`read_csv` lit des données séparées par des virgules. `StringIO` permet de traiter le texte ci-dessous comme un fichier : l’exemple fonctionne donc sans téléchargement. Avec un vrai fichier, la même opération serait `pd.read_csv("sales.csv")`.

In [ ]:
from io import StringIO

csv_text = """order_id,country,product,quantity
101,Canada,Notebook,4
102,Canada,Pen,10
103,Canada,Notebook,6
104,France,Notebook,3
105,France,Pen,8
106,France,Pen,2
"""
# Transformez le texte en objet utilisable comme un fichier, puis lisez-le comme une table CSV.
csv_file = StringIO(csv_text)
sales = pd.read_csv(csv_file, skipinitialspace=True)
display(sales)

#### Six lignes de commandes → quatre totaux → une table de comparaison
Regroupez d’abord par pays **et** par produit, puis additionnez les quantités. `as_index=False` conserve les étiquettes de regroupement comme colonnes ordinaires dans ce résumé.

Un pivot place une étiquette sur les lignes et une autre sur les colonnes. La fonction `pivot` exige une seule valeur par paire ligne/colonne. Nos commandes brutes répètent certaines paires pays/produit : `pivot_table(..., aggfunc="sum")` les combine donc. La somme répond à la question **combien d’articles ont été commandés**, pas combien de commandes ont été passées.

**Prédiction :** le total de cahiers pour le Canada est `4 + 6 = 10`. Complétez les trois autres totaux avant d’exécuter le code. Chaque paire pays/produit est présente dans cet exemple.

In [ ]:
# Créez un groupe pour chaque paire pays/produit, puis additionnez ses quantités.
order_groups = sales.groupby(["country", "product"], as_index=False)
grouped_quantities = order_groups["quantity"]
order_totals = grouped_quantities.sum()

by_country = sales.pivot_table(index="country", columns="product",
                               values="quantity", aggfunc="sum")
display(order_totals)
display(by_country)
assert by_country.loc["Canada", "Notebook"] == 10

In [ ]:
ax = by_country.plot.bar(figsize=(6, 3.2), rot=0,
                        color=["#0072B2", "#D55E00"])
ax.set(xlabel="Country", ylabel="Items ordered",
       title="A pivot table becomes a grouped bar chart")
ax.legend(title="Product")
plt.tight_layout()
plt.show()

**À essayer :** créez un nouveau pivot avec les produits sur les lignes et les pays sur les colonnes, en conservant `aggfunc="sum"`. Tracez ensuite cette table. Quelles étiquettes passent dans la légende ? **Vérification :** le total de toutes les cellules reste de `33` articles.

In [ ]:
# Build a product-by-country pivot table and plot it.

### 4. Repérer un enregistrement en double
`pd.concat` combine des tables. Ici, nous ajoutons volontairement une copie exacte de la première commande ; `ignore_index=True` donne un nouvel index de lignes à la table combinée.

`duplicated()` signale par défaut les copies d’une ligne qui apparaissent après sa première occurrence. `drop_duplicates()` conserve la première occurrence. Nous savons que cette ligne supplémentaire répète la même commande, car nous venons de la copier, y compris son `order_id`. Deux commandes différentes peuvent avoir les mêmes quantités : des mesures identiques ne constituent donc pas à elles seules une erreur.

**Prédisez, puis vérifiez :** combien de lignes et d’articles y aura-t-il avant et après la suppression du doublon ? Expliquez pourquoi la quantité totale change.

In [ ]:
# La liste [0] conserve la première ligne sous forme de DataFrame, prêt à être ajouté.
first_order = sales.iloc[[0]]
repeated = pd.concat([sales, first_order], ignore_index=True)

duplicate_mask = repeated.duplicated()
duplicate_rows = repeated.loc[duplicate_mask]
display(duplicate_rows)
deduplicated = repeated.drop_duplicates()
print("Rows before / after:", len(repeated), len(deduplicated))
total_before = repeated["quantity"].sum()
total_after = deduplicated["quantity"].sum()
print("Total quantity before / after:", total_before, total_after)
assert len(deduplicated) == 6
assert total_after == 33

## Vérifiez votre travail
Essayez les tâches avant d’ouvrir les solutions. Les cellules de pratique incomplètes ne sont pas utilisées par les exemples résolus : le notebook peut donc quand même s’exécuter depuis le début.

<details>
<summary>Exercice E · Solution de référence</summary>

```python
virginica_mask = df["species"] == "virginica"
wide_petal_mask = df["petal width (cm)"] > 2.0
# & conserve les lignes où les deux conditions sont True.
exercise_mask = virginica_mask & wide_petal_mask
wide_virginica = df.loc[exercise_mask]
mean_length = wide_virginica["petal length (cm)"].mean()
exercise_groups = df.groupby("species")
exercise_petals = exercise_groups[petal_columns]
exercise_means = exercise_petals.mean()
display(wide_virginica.head())
print("Rows:", len(wide_virginica), "Mean length:", mean_length)
display(exercise_means)
assert len(wide_virginica) == 23
assert np.isclose(mean_length, 5.760869565217392)
assert exercise_means.shape == (3, 2)
```

</details>

<details>
<summary>Exercice F et histogramme · Exemples d’observations</summary>

Dans le nuage de points des pétales, setosa occupe la zone en bas à gauche, tandis que versicolor et virginica se chevauchent. Dans ce dataset, les fleurs aux pétales plus longs ont aussi tendance à avoir des pétales plus larges. Le graphique seul ne mesure pas la capacité d’un modèle à prédire correctement l’espèce de fleurs jamais vues.

Avec la longueur du sépale sur l’axe horizontal, les espèces se chevauchent davantage le long de cet axe. Changer les bins d’un histogramme réaffiche les mêmes 150 valeurs dans des intervalles différents ; cela ne change ni l’échantillon ni sa moyenne. Regrouper les espèces dans un histogramme masque l’espèce à laquelle appartient chaque observation.

</details>

<details>
<summary>Petits problèmes sur une table · Solution de référence</summary>

```python
practice = mini.copy()
practice.loc["flower_b", "petal_width_cm"] = np.nan
missing_width_mask = practice["petal_width_cm"].isna()
missing_flower = practice.loc[missing_width_mask]
longest_first = practice.sort_values("petal_length_cm", ascending=False)
display(missing_flower)
display(longest_first)
practice.loc["flower_b", "petal_width_cm"] = 1.4
display(practice)
assert missing_flower.index.tolist() == ["flower_b"]
assert longest_first.index.tolist() == ["flower_c", "flower_b", "flower_a"]
missing_after = practice["petal_width_cm"].isna()
assert missing_after.sum() == 0
```

</details>

<details>
<summary>Défi du pivot · Solution de référence</summary>

```python
by_product = sales.pivot_table(index="product", columns="country",
                               values="quantity", aggfunc="sum")
display(by_product)
ax = by_product.plot.bar(rot=0, figsize=(6, 3.2),
                        color=["#0072B2", "#D55E00"])
ax.set(xlabel="Product", ylabel="Items ordered", title="The same totals, rearranged")
ax.legend(title="Country")
plt.tight_layout()
plt.show()
# Additionnez d’abord chaque colonne de pays, puis les totaux des pays.
country_totals = by_product.sum()
grand_total = country_totals.sum()
assert grand_total == 33
```

Le Canada et la France apparaissent maintenant dans la légende. Les quatre totaux sont toujours `10`, `10`, `3` et `10` ; réorganiser la table ne change pas les données.

</details>

## Sources et exercices supplémentaires
- [100 pandas puzzles d’ajcr](https://github.com/ajcr/100-pandas-puzzles) : légère inspiration tirée des questions **DataFrame basics**, surtout **4–11, 13 et 15–17**. Les questions sur la petite table utilisent ici nos propres données de fleurs et formulations. La leçon principale reste à un niveau introductif.

Références : [dataset Iris et provenance](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html), [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html), [données manquantes](https://pandas.pydata.org/docs/user_guide/missing_data.html), [tables pivot](https://pandas.pydata.org/docs/user_guide/reshaping.html) et [graphiques avec pandas](https://pandas.pydata.org/docs/user_guide/visualization.html).
Lecture : *Python Data Science Handbook*, Data Manipulation with Pandas et Visualization with Matplotlib.